  Creating a Youtube chatbot using OpenAI API, RAG pipeline and Youtube API

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# necessary imports for building a parallel runnable 
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [2]:
# requirements
api_key = # your openapi key here

# 1. Indexing (Ingestion of Data)

In [3]:
# youtube Video ID
video_id = '11Y3B33oCLE'

try :
    #getting the transcript of the video
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    # into a single string
    transcript_text = " ".join([i['text'] for i in transcript.to_raw_data()])
    print("Transcript fetched successfully.")
except Exception as e:
    print(f"An error occurred: {e}")
    transcript = None

Transcript fetched successfully.


In [4]:
#checking transcript raw text
transcript.to_raw_data()[:5]  # Displaying first 5 entries for brevity

[{'text': '[applause]', 'start': 1.309, 'duration': 2.02},
 {'text': '>> My relationship with you started here.',
  'start': 4.28,
  'duration': 4.4},
 {'text': 'And many of you, many of you, many of my',
  'start': 9.16,
  'duration': 4.8},
 {'text': 'friends and partners here in Taiwan,',
  'start': 11.72,
  'duration': 6.08},
 {'text': 'your companies started here.', 'start': 13.96, 'duration': 5.68}]

In [5]:
#text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=0, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.create_documents([transcript_text])

In [6]:
chunks[:5]

[Document(metadata={}, page_content='[applause] >> My relationship with you started here. And many of you, many of you, many of my friends and partners here in Taiwan, your companies started here. This is in a lot of ways the beginning of the modern computer industry, 40 years now.'),
 Document(metadata={}, page_content='Nvidia is 33 years old. The PC industry was already starting to get to Windows 1 and Windows 2 and Apple Apple 1 and Apple 2 and by the time that we came along, Windows 3.1 was the PC. And as you know, Windows 95 made PC personal. It took PC from'),
 Document(metadata={}, page_content='enterprises, com- companies, and made it into a consumer electronics device. Everybody should have one, and everybody does. This is the beginning. This computing platform did several things incredibly smart. Windows was not just disaggregated, as'),
 Document(metadata={}, page_content='you know. Windows was properly abstracted. It was architected just right. Systems BIOSes, open chipsets

In [7]:
# creating embeddings and storing into vectorstore
embeddings_gen = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview", api_key=api_key)
vec_store = FAISS.from_documents(chunks, embeddings_gen)

In [8]:
#ids for every document in the vectorstore
vec_store.index_to_docstore_id

{0: 'a314afad-9c92-45f6-a476-9f981d598151',
 1: '251fd673-b33a-47c1-ae0b-73716a67bf9d',
 2: '8862cdf6-2282-4637-b9e4-245c69f670aa',
 3: 'dbdf0a16-7f26-4390-8021-3d574c8edac1',
 4: '4a27d5d9-6d05-4172-a1de-af4f8434c771',
 5: 'b51603ed-e82d-4243-865f-c50351f1be35',
 6: '01e82e5d-8ed8-44d6-ae13-e295d1654663',
 7: '1bea7aca-dd3a-4a23-998e-4117239a1915',
 8: '437777f5-2d57-4f6b-9b92-6008c7c14c9e',
 9: 'c41af475-baad-4e2c-a39c-8ab2611c80b5',
 10: '153320e4-2463-498d-885c-91f1d512bc35',
 11: 'eac2ecbc-86c0-4e20-a827-5c08a8883408',
 12: 'df0cf46f-0816-452d-b0c4-ddf0ba298543',
 13: 'e29e009a-e732-4e67-a72e-84070266c61c',
 14: '078b8bc4-3c80-4d51-90da-e9c05c9b43e2',
 15: 'de9798e5-46c5-42ec-9eab-7dfdacba3642',
 16: '91a9522a-0dd6-4e9a-a234-2384ec692822',
 17: 'dab9faed-2f19-4ba8-9b69-bbad26c2bf86',
 18: '18f14c63-500b-4435-9238-ff4db224f8e9',
 19: '7619435f-af7c-4b08-bfa7-90cdf32e196e',
 20: '59fba408-68ee-4d6d-b02f-c2db67b5bd8e',
 21: 'b8083085-55e1-4374-ad7a-6686a809ecb7',
 22: '0c1bb85c-4fb8-

In [9]:
# let's check a document by its id
vec_store.get_by_ids([vec_store.index_to_docstore_id[0]])  # Get the first document

[Document(id='a314afad-9c92-45f6-a476-9f981d598151', metadata={}, page_content='[applause] >> My relationship with you started here. And many of you, many of you, many of my friends and partners here in Taiwan, your companies started here. This is in a lot of ways the beginning of the modern computer industry, 40 years now.')]

# 2. Retrieval

In [10]:
# careating a retriever from the vectorstore
retriever = vec_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [11]:
retriever.invoke('Nvidia Spark')

[Document(id='eac2ecbc-86c0-4e20-a827-5c08a8883408', metadata={}, page_content='in a world of agents? Agents running natively, connected to models local or [music] in the cloud, our personal AI sandboxed for security, running continuously, getting work done. The chips and the OS must evolve. Introducing RTX Spark. Everything'),
 Document(id='078b8bc4-3c80-4d51-90da-e9c05c9b43e2', metadata={}, page_content="This is the dawn of a new personal computing revolution. And it starts with NVIDIA RTX Spark. >> [music] [applause] >> Here it is. Of course, I got to show you the most beautiful part, which is video games. It is It's also the closest to our heart."),
 Document(id='1e9a8799-88f3-45b4-9d3c-f2b09de26b89', metadata={}, page_content='generative AI [music] with the Flux 2 model, makes them photo real. Multiple viewpoints, lighting conditions. What was once a complex [music] workflow is now guided and simplified by my agent. Working with [music] me on RTX Spark. Design at the'),
 Document(

# 3. Augmentation (creating Prompt Template  = (Query + Retrieved data))

In [12]:
# creating a LLM model instance
LLM = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_version="v1", api_key = api_key)

In [13]:
#creating a prompt template
PROMPT = PromptTemplate(
    template = """You are a helpful assistant that answers questions based on the context provided from a YouTube video transcript.If the context is not sufficient , please respond with "I don't know.". {context} Question: {question}""",
    input_variables = ["context", "question"]
    )

In [20]:
# creating question
question = "what about the battery life?"
retrieved_docs = retriever.invoke(question)

#joining the retrieved documents into a single context string
context = " ".join([doc.page_content for doc in retrieved_docs])

#creating final prompt
final_prompt = PROMPT.invoke({"context": context, "question": question})

#let's check how the final prompt looks
print(final_prompt)

text='You are a helpful assistant that answers questions based on the context provided from a YouTube video transcript.If the context is not sufficient , please respond with "I don\'t know.". happen to that PC when it has an autonomous agent? An agent that\'s helping you, that understands you. You could talk to it. It could look at you. You could ask it to read files, go help you do some research. It could do a lot more that I\'ll show Everything associated with CUDA, all the physics, all the biology, all the genomics, all the AI, no problem. All the computer graphics, no problem. Every single application Nvidia has ever created and every single application that Windows has ever the reinvention of the phone into what we now know as the smartphone. And so this is the beginning of that journey this is the beginning of a new line and so we have a roadmap for this. This is a brand new product family for us. Every single in a world of agents? Agents running natively, connected to models loc

# 4. Generation (response generation from LLM by passing final_prompt to the LLM)

In [21]:
response = LLM.invoke(final_prompt)
print("Response from LLM:", response.content)

Response from LLM: I don't know.


# 5. Building chain

In [ ]:
# necessary imports for building a parallel runnable 
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [27]:
# creating a function for formatting retrieved documents
def format_retrieved_docs(docs):
    return " ".join([doc.page_content for doc in docs])

In [29]:
# parallel chain to retrieve documents and format them in parallel
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_retrieved_docs),
    'question': RunnablePassthrough()
})

In [ ]:
# checking the parallel chain with a sample question
parallel_chain.invoke('Unified memory architecture')

{'context': "unified memory, TSMC 3-nanometer process, 70 billion transistors. And in close [music] collaboration with Microsoft, a Windows platform for agents. We're reinventing [music] the personal computer for creating, >> [music] >> for gaming, for agents. Everything associated with CUDA, all the physics, all the biology, all the genomics, all the AI, no problem. All the computer graphics, no problem. Every single application Nvidia has ever created and every single application that Windows has ever we've learned [music] over 33 years distilled into one chip. Blackwell [music] RTX GPU with 6,144 CUDA cores, one petaflop of AI performance, a custom 20-core [music] Grace CPU built in partnership with MediaTek, fused by [music] NVLink, 128 GB of generation of architecture we will have a desktop a laptop, a workstation and then a desktop, a laptop and workstation and the thing that I'm just incredibly pleased, incredibly honored is that 100% of the world's PC industry has joined us to 

In [31]:
# creatiing a ouptput parser for the final response
output_parser = StrOutputParser()

#final chain
chatbot = parallel_chain | PROMPT | LLM | output_parser

In [32]:
#let's ask question
chatbot.invoke("summarize the video in a few lines.") 

'The video discusses reinventing the PC for the age of AI, focusing on a "modern application, an agent." This involves a new Windows platform developed in collaboration with Microsoft, featuring unified memory, TSMC 3-nanometer process, and 70 billion transistors. The goal is to create a personal computer for creating, gaming, and agents, with autonomous agents that can help, understand, read files, and do research.'